In [ ]:
# ----------------------------------- import --------------------------------#
import matplotlib.pyplot as plt
import datetime
import os
import re
from glob import glob
import joblib
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ----------------------------------- cfg --------------------------------#
def display_dfs_with_optional_captions(
    dfs,
    captions=None,
    ncols=2,
    table_font_family='Microsoft YaHei',
    table_font_size='11pt',
    table_text_align='left',
    caption_font_family='Microsoft YaHei',
    caption_font_size='12pt',
    caption_text_align='center',
    gap='15px'
):
    htmls = []
    if captions is None:
        captions = [None] * len(dfs)
    else:
        captions = list(captions) + [None] * (len(dfs) - len(captions))

    for caption, df in zip(captions, dfs):
        styled = df.style.set_table_styles([
            {'selector': 'th', 'props': [
                ('font-family', table_font_family),
                ('font-size', table_font_size),
                ('text-align', table_text_align),
            ]},
            {'selector': 'td', 'props': [
                ('font-family', table_font_family),
                ('font-size', table_font_size),
                ('text-align', table_text_align),
            ]},
        ])
        table_html = styled.render()

        caption_html = ''
        if caption:
            caption_html = f'''
                <div style="font-family:{caption_font_family}; font-size:{caption_font_size}; text-align:{caption_text_align}; margin-bottom:6px;">
                    {caption}
                </div>
            '''

        block_html = f'''
            <div style="margin-bottom: 20px;">
                {caption_html}
                {table_html}
            </div>
        '''
        htmls.append(block_html)

    container_style = f"""
        display: grid;
        grid-template-columns: repeat({ncols}, 1fr);
        gap: {gap};
    """
    container_html = f'<div style="{container_style}">' + ''.join(htmls) + '</div>'
    display(HTML(container_html))


In [ ]:
# ----------------------------------- func --------------------------------#
def calc_lgbmodel_imp(model):
    for i, type_ in enumerate(['gain']):
        feas_imp_ = pd.DataFrame(
            model.feature_importance(importance_type=type_),
            index=model.feature_name(),
            columns=[type_]
        )
        if i == 0:
            feas_imp = feas_imp_
        else:
            feas_imp = feas_imp.merge(feas_imp_, left_index=True, right_index=True)
        feas_imp.index.name = 'features'
    return feas_imp


def get_pivot_table(data, index, columns, values, aggfunc, dropna=False,
                    margins=False, color_column=None):
    if isinstance(index, str):
        index = [index]
    if isinstance(columns, str):
        columns = [columns]
    if isinstance(values, str):
        values = [values]

    df = data[index + columns + values]
    if not dropna:
        for fea in df.columns:
            if df[fea].isnull().any():
                df_fillna(df, fea, fill_value='NaN', inplace=True)

    st = pd.pivot_table(df, index=index, columns=columns,
                        values=values, fill_value=0, dropna=dropna,
                        aggfunc=aggfunc, observed=False, margins=margins)

    if color_column is not None:
        try:
            st = st.style.background_gradient(cmap='RdYlGn', low=0.7, high=0, vmin=0, subset=color_column)
        except KeyError:
            pass
    return st


def calc_ks_auc(all_target, predicted):
    fpr, tpr, thresholds = roc_curve(all_target, predicted)
    roc_auc = round(auc(fpr, tpr), 4)
    ks = max(tpr - fpr)
    return fpr, tpr, thresholds, ks, roc_auc


def calc_perf_(df, target, fea, percentile=[95]):
    perf_list = []
    perf_list.append(df.shape[0])
    mask = (df[fea] > 0) & (pd.notnull(df[fea]))
    perf_list.append(round(mask.sum() / perf_list[0] * 100, 2))
    data = df[mask][[fea, target]].reset_index(drop=True)
    ks_, auc_ = calc_ks_auc(data[target], data[fea])[3:]
    perf_list.extend([round(ks_, 4), round(auc_, 4)])
    if percentile is not None:
        for p in percentile:
            thred = np.percentile(data[fea], p)
            perf_list.append(round(data[data[fea] > thred][target].mean() / data[target].mean(), 3))
    return perf_list


def calc_perf(df, target, fea, gp_col):
    perf_list = []
    p_list = []
    for p in sorted(df[gp_col].unique()):
        p_list.append(p)
        perf_list.append(
            calc_perf_(
                df[df[gp_col] == p][[target, fea]].reset_index(drop=True),
                target, fea, percentile=[95]
            )
        )
    return p_list, np.array(perf_list)


def calc_psi_fast(ref_data, data, feas, cut_params=10, display_=False):
    try:
        bins = pd.qcut(ref_data[feas], cut_params, duplicates='drop', retbins=True)[1].tolist()
    except IndexError:
        unique = ref_data[feas].unique()
        if len(unique) > 1:
            bins = [unique[~np.isnan(unique)][0]]
        else:
            bins = [-np.inf, np.inf]

    if bins != [-np.inf, np.inf]:
        bins = [-np.inf] + bins + [np.inf]

    need = (
        pd.DataFrame(pd.cut(ref_data[feas], bins).value_counts(dropna=False, normalize=True))
        .merge(
            pd.DataFrame(pd.cut(data[feas], bins).value_counts(dropna=False, normalize=True)),
            left_index=True, right_index=True, how='outer'
        )
    )
    need.replace([0, np.nan], 0.01, inplace=True)
    if display_:
        display.display(need)

    return pd.DataFrame({
        'ref_data_cnt': ref_data.shape[0],
        'observe_data': data.shape[0],
        'psi': sum((need.iloc[:, 1] - need.iloc[:, 0]) * np.log(need.iloc[:, 1] / need.iloc[:, 0]))
    }, index=[feas])


def calc_monitor_stats_bywindow(data, by, fea, cut_params=10, step=1):
    tmp = data[[by, fea]]
    by_list = sorted(tmp[by].unique())
    stats_all = pd.DataFrame()

    for i in range(len(by_list) - 1):
        ref_by, obs_by = by_list[i: i + step], by_list[i + step: i + 2 * step]
        if len(ref_by) > 0 and len(obs_by) > 0:
            ref_data = tmp[tmp[by].isin(ref_by)].reset_index(drop=True)
            obs_data = tmp[tmp[by].isin(obs_by)].reset_index(drop=True)
            try:
                stats = calc_psi_fast(ref_data, obs_data, fea, cut_params)
            except:
                stats = pd.DataFrame({
                    'ref_data_cnt': ref_data.shape[0],
                    'observe_data': obs_data.shape[0],
                    'psi': 1
                }, index=[fea])
            stats['data_description'] = f'{ref_by} vs {obs_by}'
            stats = stats.reindex(columns=[list(stats)[-1]] + list(stats)[:-1])
            stats_all = pd.concat([stats_all, stats])

    return stats_all


def LogitTransformation(prob):
    if prob <= 0 or prob >= 1:
        return 0
    return np.log(prob / (1 - prob))


def get_valid_prob(x):
    if x <= 0 or x >= 1:
        return 0
    return 1


In [ ]:
# ----------------------------------- plot --------------------------------#
def df_fillna(data, fea, fill_value='NaN', inplace=False):
    df = data[[fea]]
    if re.search('category', str(df[fea].dtypes)):
        try:
            df[fea] = df[fea].cat.add_categories(fill_value)
        except ValueError:
            pass
    df[fea] = df[fea].fillna(fill_value)
    if inplace:
        data[fea] = df[fea]
        return None
    else:
        return df[fea]


def _get_state(data, feas, target, sort_feas=None):
    aggfunc = {feas: len}
    if isinstance(target, str):
        target = [target]
    if target is not None:
        aggfunc.update(dict(zip(target, ['mean'] * len(target))))
    if str(data[feas].dtypes) == 'category':
        if sum(pd.isnull(data[feas])) > 0:
            df_fillna(data, feas, fill_value='NaN', inplace=True)
    if sort_feas is None:
        state = data.groupby([feas], dropna=False).agg(aggfunc).rename(columns={feas: 'count'})
    else:
        state = data.sort_values(sort_feas).groupby([feas], dropna=False, sort=False).agg(aggfunc).rename(columns={feas: 'count'})
    state['count_rto'] = state['count'] / data.shape[0]
    if target is not None:
        for target_ in target:
            state[f'{target_}_mean'] = data[target_].mean()
    return state


def plt_plot_twinx(state, cols_name, target, title='feas',
                   color_bar=None, color_line=None, scale_num=20, save_path=None):
    color_bar = color_bar or []
    color_line = color_line or []

    state.index = state.index.astype(str)
    fig = plt.figure(figsize=[15, 8], clear=True)

    if isinstance(cols_name, str):
        cols_name = [cols_name]
    if isinstance(target, str):
        target = [target]

    length = len(cols_name)
    bar_width = 0.8 / length
    x = np.arange(len(state))
    ax1 = fig.add_subplot(111)
    ax = plt.gca()
    ax.locator_params("x")
    ax.tick_params(labelsize=16, rotation=0)

    for i, name in enumerate(cols_name):
        ax1.bar([c + i * bar_width for c in x], state[name].tolist(), bar_width,
                color=color_bar[i] if i < len(color_bar) else None, alpha=0.8, label=name)

    ax2 = plt.twinx()
    for i, name in enumerate(target):
        ax2.plot([c + min(0.2 * (length - 1), 0.3) for c in x], state[name].tolist(),
                 alpha=1, linewidth=2,
                 color=color_line[i] if i < len(color_line) else None,
                 label=name)

    ax2.tick_params(labelsize=16)
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc='upper right', fontsize=20)

    x_ticks_pos = [c + min(0.2 * (length - 1), 0.3) for c in x]
    loc = np.quantile(np.arange(len(x_ticks_pos)), np.linspace(0, 1, scale_num + 2)).tolist()
    if state.index[-1] == 'nan':
        loc.append(len(x_ticks_pos) - 2)
    plt.xticks(ticks=[x_ticks_pos[int(i)] for i in loc], labels=[state.index[int(i)] for i in loc])
    plt.tick_params(labelsize=16, rotation=0)
    plt.title(title, fontsize=20)

    if save_path is not None:
        plt.savefig(join(save_path, f'{title}.png'), bbox_inches='tight')

    plt.close(fig)
    return fig


<div style="font-family:'Microsoft YaHei', 微软雅黑, sans-serif; font-size:25pt; text-align:center;">
小雨点数科模型评估报告
</div>

<div style="font-family:'Microsoft YaHei', 微软雅黑, sans-serif; font-size:15pt; text-align:left;">
一. 特征重要性：
</div>

In [ ]:
mdl_paths = glob('./model/*_model.pkl')

In [ ]:
dfs = []
captions = []
lr_model = []
lgb_model = []


for i, model_path in enumerate(mdl_paths):
    model = joblib.load(model_path)
    model_name = model_path.split('\\')[1].split('.pkl')[0]
    
    if re.search('lightgbm', str(model)):
        lgb_model.append(model_name)
        imp_head5 = calc_lgbmodel_imp(model).sort_values('gain', ascending=False).reset_index()[:5]
        captions.append(f'{model_name}高重要性特征如下:')
        dfs.append(imp_head5)
    else:
        lr_model.append(model_name)

In [ ]:
txt = str(lgb_model).replace("'","").replace("[","").replace("]","")
display(HTML(f'<span style="font-family:Microsoft YaHei; font-size:12pt;">{txt} 是基于单维度指标，使用LGB构造的树模型'))

txt = str(lr_model).replace("'","").replace("[","").replace("]","")
display(HTML(f'<span style="font-family:Microsoft YaHei; font-size:12pt;">{txt} 是基于各维度子分，使用LR构造的融合模型'))

In [ ]:
display_dfs_with_optional_captions(
    dfs,
    captions,
    ncols=2,
    table_font_family='Microsoft YaHei',
    table_font_size='10pt',
    table_text_align='left',
    caption_font_family='Microsoft YaHei',
    caption_font_size='12pt',
    caption_text_align='left',
    gap='10px')

In [ ]:
print()
print()

<div style="font-family:'Microsoft YaHei', 微软雅黑, sans-serif; font-size:15pt; text-align:left;">
二. 模型效果：
</div>

In [ ]:
data = pd.read_csv('df.csv')

In [ ]:
scores_all = ['model5_proba',
              'model6_proba',
              'model7_proba',
              'model9_proba']

In [ ]:
for i, type_ in enumerate(['x_train','x_test','oot']):
    df = data[data['data_type']==type_].reset_index(drop=True)
    
    # calc perf
    perf_list = []
    for score in scores_all:
        perf_list.append(calc_perf_(df, 'flag', score))
    state_perf = pd.DataFrame(perf_list)
    state_perf.index = scores_all
    state_perf.columns=['样本量','覆盖度','ks','auc','5%lift']
    state_perf['覆盖度'] = state_perf['覆盖度'].apply(lambda x: str(x)+'%')
    state_perf = state_perf.reset_index().rename(columns={'index':'score'})   
    new_cols = pd.MultiIndex.from_tuples([(f'{list(state_perf)[0]}','')]+[(f'{type_}',c) for c in list(state_perf)[1:]])
    state_perf.columns = new_cols    
    if i==0:
        out = state_perf
    else:
        out = out.merge(state_perf, on=[('score','')])

# calc psi
feas_dict = dict(zip(scores_all,[10]*len(scores_all)))
stats_psi = pd.DataFrame()
for fea, cut_params in feas_dict.items():
    stats_psi = pd.concat([stats_psi, calc_monitor_stats_bywindow(data, by='month', fea=fea, cut_params=cut_params, step=1)])
psi = stats_psi.reset_index().groupby(['index']).agg({"psi":'mean'}).reset_index().rename(columns={'index':'score'})  
new_cols = pd.MultiIndex.from_tuples([(f'{c}','') for c in list(psi)])
psi.columns = new_cols 

# merge
out = out.merge(psi)

In [ ]:
display(HTML(f'<span style="font-family:Microsoft YaHei; font-size:12pt;">1.  各模型覆盖度、ks、auc、逐月扫窗psi'))

styled_df = out.style.set_table_styles([
    {'selector': 'th', 'props': [('font-family', 'Microsoft YaHei'), ('font-size', '10pt'), ('text-align', 'left')]}
])    
display(styled_df)

In [ ]:
print('')
display(HTML(f'<span style="font-family:Microsoft YaHei; font-size:12pt;">2.  各模型逐月效果'))

In [ ]:
gp_col='month'

In [ ]:
for i, score in enumerate(scores_all):
 
    type_list, perf = calc_perf(data, 'flag', score, gp_col)
    state_gp_perf = pd.DataFrame(perf)
    state_gp_perf.index = type_list
    state_gp_perf.columns=['样本量','覆盖度','ks','auc','5%lift']
    state_gp_perf['覆盖度'] = state_gp_perf['覆盖度'].apply(lambda x: str(x)+'%')
    state_gp_perf = state_gp_perf.reset_index().rename(columns={'index':gp_col})
    state_gp_perf['score'] = score
    
    st_gp_perf = get_pivot_table(state_gp_perf, index=gp_col, columns='score', values=['样本量','ks','auc','5%lift'], 
                                 aggfunc=['mean'], dropna=False).iloc[:,::-1]
    st_gp_perf.columns = pd.MultiIndex.from_tuples([(c[2],c[1]) if c[2]!='' else (c[0],'All') for c in st_gp_perf.columns])        
    change_col = st_gp_perf.columns[0]
    new_index = pd.MultiIndex.from_arrays([st_gp_perf.index, st_gp_perf[change_col]], names=[gp_col, '样本量'])
    st_gp_perf = st_gp_perf.drop(columns=[change_col])
    st_gp_perf.index = new_index

    if i==0:
        out = st_gp_perf
    else:
        out = out.merge(st_gp_perf, left_index=True, right_index=True)

In [ ]:
styled_df = out.style.set_table_styles([
    {'selector': 'th', 'props': [('font-family', 'Microsoft YaHei'), ('font-size', '10pt'), ('text-align', 'left')]}
])    
display(styled_df)

In [ ]:
print()
display(HTML(f'<span style="font-family:Microsoft YaHei; font-size:12pt;">3.  各模型分箱后，各箱lift值'))

In [ ]:
cut_cnt = 10
font_cn = 'Microsoft YaHei'
plt.rcParams['font.family'] = font_cn

In [ ]:
for score in scores_all:
    bins = pd.qcut(data[score], cut_cnt, duplicates='drop', retbins=True)[1].tolist()
    
    for i, type_ in enumerate(['x_train','x_test','oot']):    
        df = data[data['data_type']==type_].reset_index(drop=True)
        df['cut'] = pd.cut(df[score], bins)
        st_plot = _get_state(df, 'cut', 'flag', sort_feas=None)
        st_plot['lift'] = st_plot['flag']/st_plot['flag_mean']
        st_plot.columns = [f'{type_}'+'_'+f'{c}' for c in list(st_plot)]    
        if i==0:
            out = st_plot
        else:
            out = out.merge(st_plot, left_index=True, right_index=True, how='left')
            
    out.index = [f'分箱{c}' for c in range(1,len(bins))] + out.index[len(bins)-1:].tolist()
    pic = plt_plot_twinx(out, cols_name=[c for c in out if re.search(f'count_rto$', c)], 
                   target=[c for c in out if re.search(f'lift$', c)], title=score,
                   color_bar=['steelblue','darkseagreen','tan','rosybrown','cadetblue','cadetblue'],
                   color_line=['red','brown','tomato','violet'],scale_num=20)
    display(pic)

In [ ]:
print()
print()

<div style="font-family:'Microsoft YaHei', 微软雅黑, sans-serif; font-size:15pt; text-align:left;">
三. 对base_score增益：
</div>

In [ ]:
data = pd.read_csv('df.csv')

In [ ]:
base_score = 'base_proba'
scores_all = ['model5_proba',
              'model6_proba',
              'model7_proba',
              'model9_proba']

In [ ]:
for fea in set([base_score]+scores_all):
    data[f'{fea}_proc'] = data[fea].fillna(0)
    data[f'{fea}_valid'] = data[f'{fea}_proc'].apply(lambda x: get_valid_prob(x))
    data[f'{fea}_proc'] = data[f'{fea}_proc'].apply(lambda x: LogitTransformation(x))

In [ ]:
params = {'penalty': 'l1',
         'tol': 0.0025,
         'C': 6.0,
         'solver': 'saga',
         'max_iter': 200,
         'warm_start': True,
         'random_state': 2020}

In [ ]:
for fea in [f'{fea}_proc' for fea in scores_all]:
    
    #------------------------ get feas ---------------------#      
    feas = [f'{base_score}_proc', fea]
    out_score = f"{base_score}_add_{fea.replace('_proc','')}"
    feas = feas + [f'{c.replace("proc","valid")}' for c in feas]
    

    # ------------------------ model ------------------------#
    model = LogisticRegression(**params)  
    x_train = data[data['data_type']=='x_train'].reset_index(drop=True)
    model.fit(x_train[feas], x_train['flag'])  

    # ------------------------ predict ------------------------#
    data[out_score] = model.predict_proba(data[feas])[:,1:]
    idx = data[[c for c in feas if re.search('valid',c)]].sum(axis=1)==0
    data.loc[idx,out_score] = np.nan  

data.drop(columns=[c for c in data if re.search('proc|valid',c)], inplace=True)

In [ ]:
add_scores_all = [base_score] + [f'{base_score}_add_{c}' for c in scores_all]

for i, type_ in enumerate(['x_train','x_test','oot']):
    df = data[data['data_type']==type_].reset_index(drop=True)
    
    perf_list = []
    for score in add_scores_all:
        perf_list.append(calc_perf_(df, 'flag', score))
    state = pd.DataFrame(perf_list)
    state.index = add_scores_all
    state.columns=['样本量','覆盖度','ks','auc','5%lift']
    state['覆盖度'] = state['覆盖度'].apply(lambda x: str(x)+'%')
    state = state.reset_index().rename(columns={'index':'score'})

    tmp = state.iloc[:,3:] - state.iloc[0,3:]
    tmp.columns = [f'{c}_增益' for c in tmp.columns]
    tmp = tmp.applymap(lambda x: f"{x * 100:.2f}%")
    state = state.merge(tmp, left_index=True, right_index=True).iloc[1:,[0,-3,-2,-1]]
    new_cols = pd.MultiIndex.from_tuples([(f'{list(state)[0]}','')]+[(f'{type_}',c) for c in list(state)[1:]])
    state.columns = new_cols 
    
    
    if i==0:
        out = state
    else:
        out = out.merge(state, on=[('score',    '')])
        
out.iloc[:,0] = [c.replace(f'{base_score}_add_','') for c in out.iloc[:,0].tolist()]

In [ ]:
display(HTML(f'<span style="font-family:Microsoft YaHei; font-size:12pt;">各模型与基准分融合后，对基准分的增益：'))

styled_df = out.style.set_table_styles([
    {'selector': 'th', 'props': [('font-family', 'Microsoft YaHei'), ('font-size', '10pt'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-family', 'Microsoft YaHei'), ('font-size', '10pt'), ('text-align', 'left')]}
])    
display(styled_df)

In [ ]:
print()
print()

In [ ]:
import nbformat

nb = nbformat.read("小雨点模型报告.ipynb", as_version=4)

new_cells = []
for cell in nb.cells:
    if cell.cell_type == "code":
        if cell.outputs:
            new_cell = nbformat.v4.new_code_cell(source="", outputs=cell.outputs, metadata=cell.metadata)
            new_cells.append(new_cell)
    else:
        new_cells.append(cell)

nb.cells = new_cells

nbformat.write(nb, "小雨点模型报告_唯品富邦.ipynb")

In [ ]:
! jupyter nbconvert --to html --no-input 小雨点模型报告_唯品富邦.ipynb